## Cell 1 â€” clone + install (idempotent)

Pulls the latest code from GitHub, installs dependencies into the Kaggle Python env, and registers the Atari ROMs. Safe to re-run.

In [ ]:
import os, shutil, subprocess, sys

REPO = 'spaceinvaderrl'
REPO_DIR = f'/kaggle/working/{REPO}'
CKPT_DIR = f'{REPO_DIR}/runs'

# IMPORTANT: re-cloning the repo would nuke our checkpoint directory
# (runs/) and force the next run to start from scratch. Stash any existing
# runs/ to /kaggle/working/_runs_stash BEFORE the clone, restore AFTER.
_STASH = '/kaggle/working/_runs_stash'
if os.path.isdir(_STASH):
 shutil.rmtree(_STASH)
if os.path.isdir(f'{REPO_DIR}/runs'):
 shutil.move(f'{REPO_DIR}/runs', _STASH)
if os.path.isdir(REPO_DIR):
 shutil.rmtree(REPO_DIR)

os.chdir('/kaggle/working')
!git clone https://github.com/FunctionalFalcon/{REPO}.git

os.chdir(REPO_DIR)
!git pull origin main || echo 'pull skipped (offline?)'

# Restore stashed runs/ (if any). The fresh repo has no runs/ directory,
# so this just merges the saved checkpoints back in.
if os.path.isdir(_STASH):
 if os.path.isdir(f'{REPO_DIR}/runs'):
 # shouldn't happen (repo has no runs/), but be defensive
 shutil.rmtree(f'{REPO_DIR}/runs')
 shutil.move(_STASH, f'{REPO_DIR}/runs')
 print(f'[resume] restored checkpoint directory from {_STASH}')
 else:
 shutil.move(_STASH, f'{REPO_DIR}/runs')
 print(f'[resume] restored checkpoint directory from {_STASH}')
else:
 os.makedirs(f'{REPO_DIR}/runs', exist_ok=True)

# requirements.txt pins the modern stack (gymnasium 1.3.0 / shimmy 2.0.1 /
# ale-py 0.10.2). Don't override here - older pins (gymnasium 0.29.1, ale-py
# 0.8.x) aren't on PyPI for Python 3.12 and will hard-fail the install.
!pip install --no-deps --no-cache-dir -q -r requirements.txt 2>&1 | tail -5

import gymnasium as gym
env = gym.make('ALE/SpaceInvaders-v5', frameskip=1, repeat_action_probability=0.25)
env.close()
print('envs registered OK')

## Cell 2 — smoketest (~3 min on T4)

Runs the from-scratch DQN for 5,000 steps to validate the whole pipeline before committing to the multi-hour run. Expected: a `.pt` saved under `runs/`, the eval CSV has 1 row, loss small and bounded (grad clipping caps spikes so anything < 5 is fine), no NaNs.

Pipeline: env_fixed with min_repeat=3 → Double DQN → NatureCNN → Huber loss + grad clipping → Adam.

In [ ]:
os.chdir(REPO_DIR)
!python -m scratch.train --total_steps 5000 --save_freq 5000 --eval_freq 5000

import os
assert os.path.exists(f'{CKPT_DIR}/dqn_scratch_step_5000.pt'), 'no checkpoint'
assert os.path.exists(f'{CKPT_DIR}/dqn_scratch_eval.csv'), 'no eval csv'
print('\n[smoketest] OK')


## Cell 3 — full 8,000,000-step run on T4

Wall time estimate on T4: ~2.5-3 hours at 750-1500 sps. Saves a `.pt` every 100k steps (80 checkpoints total), runs 5 greedy eval episodes every 100k steps. Auto-zips a rescue snapshot to `/kaggle/working/runs.zip` every 200k steps.

Trains against env_fixed (min_repeat=3) — the same wrapper that fixes the walk-off-screen bug from the SB3 baseline. Uses Double DQN (online picks, target values) and gradient clipping (max_grad_norm=10) on top of vanilla DQN — both single-paper upgrades, both ~2 lines.

Previous vanilla-DQN 2M-step run on this repo hit ~250 reward; expected Double DQN 8M is 600-800.

In [ ]:
os.chdir(REPO_DIR)
!python -m scratch.train --total_steps 8000000 --save_freq 100000 --eval_freq 100000

## Cell 4 — emergency resume (only if Cell 3 was interrupted)

If Kaggle disconnected you mid-run, the last completed checkpoint (`dqn_scratch_step_<N>.pt` or `dqn_scratch_crashsave.pt`) is what you resume from. The `--resume latest` flag auto-picks the most recent step checkpoint. Re-run Cell 4 until the total reaches 8,000,000.

In [ ]:
os.chdir(REPO_DIR)
!python -m scratch.train --resume latest --total_steps 8000000 --save_freq 100000 --eval_freq 100000

## Cell 5 â€” greedy evaluation (20 episodes)

Loads `runs/dqn_scratch_final.pt` and runs the from-scratch agent greedily for 20 episodes. Reports mean +/- std episode reward. Compare against the SB3 baseline (304.2 +/- 123.5) to see how close the from-scratch implementation gets.

`scratch/evaluate.py` reads `min_repeat` from the checkpoint's saved hp dict so eval env matches training env exactly.

In [ ]:
os.chdir(REPO_DIR)
!python -m scratch.evaluate --checkpoint {CKPT_DIR}/dqn_scratch_final.pt --episodes 20


## Cell 6 â€” record videos (one trained, one random baseline)

Records a 30fps MP4 of the trained agent + a random-policy baseline. Bypasses gymnasium's RecordVideo (fps=None crash on Kaggle) and uses imageio directly. The trained recorder uses env_fixed with min_repeat pulled from the checkpoint so the video matches what training saw.

In [ ]:
%%writefile /tmp/record_scratch_trained.py
"""Record a video of the FROM-SCRATCH DQN agent playing Space Invaders.

Loads a torch .pt checkpoint (not SB3's .zip), rebuilds QNetwork, and
writes an MP4 via imageio (gymnasium's RecordVideo crashes on Kaggle
because of an fps=None bug).

Uses env_fixed with min_repeat pulled from the checkpoint's saved hp
dict so the recorded env matches the training env exactly.
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

# Insert the repo root at the front of sys.path so the absolute imports
# below (`from scratch.X`, `from shared.X`) resolve regardless of CWD.
# Without this, running this script from /tmp fails with
# `ModuleNotFoundError: No module named 'scratch'`.
_REPO = Path('/kaggle/working/spaceinvaderrl')
if str(_REPO) not in sys.path:
 sys.path.insert(0, str(_REPO))

import imageio.v2 as imageio
import torch

from scratch.network import QNetwork
from scratch.hyperparam import Hyperparameters
from scratch.evaluate import greedy_action
from shared.preprocessing import env_fixed

VIDEO_DIR = _REPO / 'videos'

def main():
	p = argparse.ArgumentParser()
	p.add_argument('--checkpoint', type=str, required=True)
	p.add_argument('--episodes', type=int, default=1)
	p.add_argument('--name-prefix', type=str, default='trained-scratch')
	p.add_argument('--seed', type=int, default=42)
	p.add_argument('--fps', type=int, default=30)
	args = p.parse_args()

	VIDEO_DIR.mkdir(exist_ok=True)

	ckpt = torch.load(args.checkpoint, map_location='cpu', weights_only=False)
	hp = Hyperparameters(**ckpt.get('hp', {}))
	num_actions = int(getattr(hp, 'num_actions', 6))
	min_repeat = int(getattr(hp, 'min_repeat', 4))
	q_net = QNetwork(num_actions=num_actions)
	q_net.load_state_dict(ckpt['model_state_dict'])
	q_net.eval()

	env = env_fixed(seed=args.seed, render_mode='rgb_array', min_repeat=min_repeat)
	for ep in range(args.episodes):
		out_path = VIDEO_DIR / f'{args.name_prefix}-episode-{ep}.mp4'
		print(f' Recording episode {ep+1} -> {out_path}')
		obs, _ = env.reset()
		total, steps, done = 0.0, 0, False
		with imageio.get_writer(out_path, fps=args.fps) as writer:
			while not done:
				writer.append_data(env.render())
				action = greedy_action(q_net, obs, device='cpu')
				obs, r, term, trunc, _ = env.step(action)
				total += float(r)
				steps += 1
				done = term or trunc
		print(f' Episode {ep+1}: {steps} steps, reward={total:.1f}')
		print(f' saved: {out_path}')
	env.close()
	return 0

if __name__ == '__main__':
	sys.exit(main())

In [ ]:
import os, subprocess, sys
os.chdir(REPO_DIR)

vdir = f'{REPO_DIR}/videos'
if os.path.isdir(vdir):
 for n in os.listdir(vdir):
 full = os.path.join(vdir, n)
 if os.path.isfile(full):
 os.remove(full)

# Both scripts do absolute imports of `from scratch.X` and
# `from shared.X`. Those resolve only when CWD is the repo root
# (parent of scratch/ and shared/). Passing cwd=REPO_DIR makes
# sys.path include REPO_DIR so the imports work.
res = subprocess.run([
 sys.executable, '/tmp/record_scratch_trained.py',
 '--checkpoint', f'{CKPT_DIR}/dqn_scratch_final.pt',
 '--episodes', '1',
], cwd=REPO_DIR)
if res.returncode != 0:
 print('[record_trained] FAILED - see traceback above.')

# Random baseline: keep using make_env (no MinActionRepeat) so the
# comparison video shows raw random behavior, not random-with-commit.
res = subprocess.run([
 sys.executable, f'{REPO_DIR}/sb3/record_random.py',
 '--episodes', '1',
], cwd=REPO_DIR)
if res.returncode != 0:
 print('[record_random] FAILED - see traceback above.')

print('\n[videos] written to:', vdir)
for n in sorted(os.listdir(vdir)):
 full = os.path.join(vdir, n)
 print(f' {n} ({os.path.getsize(full):,} bytes)')

## Cell 7 â€” zip everything for download

Kaggle's UI lets you download anything in `/kaggle/working`. This zips the videos + runs + logs into a single file you can grab at the end.

In [ ]:
import os
os.chdir('/kaggle/working')
out = '/kaggle/working/runs.zip'
if os.path.exists(out):
 os.remove(out)
# Pack everything you'd want to download into one file:
# - {REPO}/videos: .mp4 recordings from Cell 6 (may not exist if Cell 6
#   didn't run; zip -rq will skip with no error)
# - {REPO}/runs: step_*.pt, final.pt, eval.csv, metrics.csv, train.log
# We do NOT include the rescue runs.zip at the working dir root to avoid
# zip-into-self errors (it's a duplicate of the curated content above
# anyway, plus 5 most-recent step ckpts + CSVs).
!zip -rq {out} {REPO}/videos {REPO}/runs 2>&1 | tail -5 || true
print(f'\n[done] {out} ({os.path.getsize(out):,} bytes)')
print('Download this file via the Kaggle file browser to keep your trained model.')